# Tugas — Studi Ablasi RNN, LSTM, GRU vs ANN

**Workshop Deep Learning · Teknik Telekomunikasi ITERA · Tenggat: Rabu, 28 Oktober 2026, 23.59 WIB**

Di Sesi 2 kita membandingkan ANN, RNN, LSTM, dan GRU dengan **satu** konfigurasi. Pertanyaan tugas ini: **seberapa besar hasil itu bergantung pada pilihan hyperparameter?** Kalian akan mengubah **satu faktor dalam satu waktu** dari konfigurasi acuan (*studi ablasi*), mencari konfigurasi terbaik memakai data **validasi**, lalu menguji apakah konfigurasi itu juga bagus di lokasi lain di Milan.

| Bagian | Isi | Status |
|---|---|---|
| 0 | Konfigurasi acuan & baseline naif | wajib |
| A | Ablasi 5 faktor: jendela W, ukuran hidden, jumlah lapisan, *learning rate*, kelompok fitur | wajib |
| B | Ablasi **minimal 2** faktor tambahan pilihan kelompok | wajib (pilih) |
| C | Konfigurasi terbaik per model (dipilih dengan **val MAE**) → uji di minggu test & Natal | wajib |
| D | Uji generalisasi di **lokasi kelompok** | wajib |

**Aturan main**

1. Jalankan sel dari atas ke bawah. Fungsi inti (data, model, training) **jangan diubah**, kecuali untuk faktor Bagian B yang memang membutuhkannya. Bila mengubah, jelaskan di laporan.
2. Semua run otomatis dicatat ke `hasil/ablasi.csv`. Run yang sudah ada **dilewati**, jadi notebook aman dijalankan ulang bila terputus (misalnya di Colab).
3. Setiap konfigurasi dilatih dengan **3 seed**. Laporkan **rata-rata ± simpangan baku**, bukan hasil satu run.
4. Pilih konfigurasi terbaik dengan **val MAE**. Minggu test & Natal hanya untuk pelaporan akhir. Memilih berdasarkan test = mengintip kunci jawaban.
5. Commit notebook **beserta output-nya** dan seluruh isi folder `hasil/` ke repo kelompok.

In [ ]:
import json, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

# ---- ISI DULU ----
LOKASI_KELOMPOK = None    # kode 0–9 = digit terakhir NIM terkecil di kelompok, contoh: 7

# ---- Pengaturan umum ----
CEPAT = False             # True = 1 seed & 20 epoch, HANYA untuk mencoba kode. Hasil akhir wajib CEPAT = False.
SEEDS = [0] if CEPAT else [0, 1, 2]
MODELS = ["ANN", "RNN", "LSTM", "GRU"]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
HASIL_DIR = Path("hasil")
HASIL_DIR.mkdir(exist_ok=True)
CSV_HASIL = HASIL_DIR / ("ablasi_cepat.csv" if CEPAT else "ablasi.csv")

NAVY, GOLD, RED, TEAL, GRAY = "#0B1F3A", "#D4A017", "#C8102E", "#1E8C8C", "#5A5A5A"
WARNA_MODEL = {"ANN": GOLD, "RNN": RED, "LSTM": NAVY, "GRU": TEAL}
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25})

assert LOKASI_KELOMPOK in range(10), "Isi LOKASI_KELOMPOK dengan kode lokasi kelompok (0–9)"
print("PyTorch", torch.__version__, "| device:", DEVICE, "| hasil ->", CSV_HASIL)

---
## 1. Data

- `data/milano_duomo_5060.csv`: kotak Duomo, sama dengan Sesi 2 (dipakai di Bagian 0–C).
- `data/milano_tugas_10_lokasi.csv`: 10 lokasi lain (kolom `kode` 0–9), dipakai di Bagian D.

Pembagian waktu sama dengan Sesi 2: train 11 Nov – 8 Des, val 9 – 15 Des, test 16 – 22 Des, Natal 23 – 29 Des. Minggu pertama (4 – 10 Nov) hanya menjadi konteks jendela.

Kelompok fitur yang bisa dipilih (faktor A5):

| Nama | Fitur |
|---|---|
| `net` | net |
| `net+call+sms` | net, call, sms |
| `net+waktu` | net, jam (sin, cos), hari (sin, cos) |
| `semua` | net, call, sms, jam (sin, cos), hari (sin, cos) — **acuan** |

In [ ]:
LOKASI = pd.read_csv("data/milano_tugas_10_lokasi.csv", usecols=["kode", "lokasi", "kotak"]).drop_duplicates().set_index("kode")
WAKTU = ["jam_sin", "jam_cos", "hari_sin", "hari_cos"]
KELOMPOK_FITUR = {
    "net": ["net"],
    "net+call+sms": ["net", "call", "sms"],
    "net+waktu": ["net"] + WAKTU,
    "semua": ["net", "call", "sms"] + WAKTU,
}
TARGET = "net"

def muat_data(lokasi):
    """lokasi = "duomo" atau kode 0–9. Mengembalikan DataFrame per jam dengan 7 kolom fitur."""
    if lokasi == "duomo":
        df = pd.read_csv("data/milano_duomo_5060.csv", parse_dates=["waktu"])
    else:
        df = pd.read_csv("data/milano_tugas_10_lokasi.csv", parse_dates=["waktu"])
        df = df[df.kode == lokasi].drop(columns=["kode", "lokasi", "kotak"])
    df = df.set_index("waktu")
    jam, hari = df.index.hour.values, df.index.dayofweek.values
    df["jam_sin"], df["jam_cos"] = np.sin(2 * np.pi * jam / 24), np.cos(2 * np.pi * jam / 24)
    df["hari_sin"], df["hari_cos"] = np.sin(2 * np.pi * hari / 7), np.cos(2 * np.pi * hari / 7)
    return df

def nama_lokasi(lokasi):
    return "Duomo" if lokasi == "duomo" else LOKASI.loc[lokasi, "lokasi"]

LOKASI.assign(kelompok_kalian=lambda d: np.where(d.index == LOKASI_KELOMPOK, "<--", ""))

In [ ]:
_CACHE = {}

def siapkan(lokasi, fitur):
    """Standarisasi dengan statistik sebelum minggu validasi (tidak mengintip masa depan)."""
    kunci = (lokasi, fitur)
    if kunci not in _CACHE:
        df = muat_data(lokasi)
        kolom = KELOMPOK_FITUR[fitur]
        t = df.index
        split = {
            "train": (t >= "2013-11-11") & (t < "2013-12-09"),
            "val":   (t >= "2013-12-09") & (t < "2013-12-16"),
            "test":  (t >= "2013-12-16") & (t < "2013-12-23"),
            "natal": (t >= "2013-12-23"),
        }
        sebelum_val = t < "2013-12-09"
        mean, std = df.loc[sebelum_val, kolom].mean(), df.loc[sebelum_val, kolom].std()
        Z = ((df[kolom] - mean) / std).values.astype(np.float32)
        _CACHE[kunci] = dict(Z=Z, net=df[TARGET].values, split=split, mean=mean[TARGET], std=std[TARGET],
                             i_target=kolom.index(TARGET), waktu=t)
    return _CACHE[kunci]


def buat_jendela(d, W, nama_split, H):
    """Titik awal i: X = Z[i-W:i], y = net terstandarisasi jam i … i+H-1.
    Sampel masuk split bila jam target terakhirnya (i+H-1) ada di split tersebut."""
    awal = np.arange(W, len(d["Z"]) - H + 1)
    awal = awal[d["split"][nama_split][awal + H - 1]]
    X = np.stack([d["Z"][i - W:i] for i in awal])
    y = np.stack([d["Z"][i:i + H, d["i_target"]] for i in awal])
    return torch.tensor(X), torch.tensor(y), awal


def metrik(y_asli, y_pred):
    e = y_pred - y_asli
    return {"MAE": float(np.mean(np.abs(e))), "RMSE": float(np.sqrt(np.mean(e ** 2))),
            "MAPE": float(np.mean(np.abs(e) / y_asli) * 100)}


def baseline_naif(lokasi, H=1):
    """MAE/RMSE/MAPE tiga tebakan tanpa belajar, di split val, test, dan natal."""
    d = siapkan(lokasi, "semua")
    baris = []
    for nama_split in ["val", "test", "natal"]:
        _, _, awal = buat_jendela(d, 168, nama_split, H)
        asli = d["net"][awal[:, None] + np.arange(H)]
        tebakan = {
            "Naif: nilai terakhir": np.repeat(d["net"][awal - 1][:, None], H, axis=1),
            "Naif: kemarin": d["net"][awal[:, None] + np.arange(H) - 24],
            "Naif: minggu lalu": d["net"][awal[:, None] + np.arange(H) - 168],
        }
        for nama, pred in tebakan.items():
            baris.append({"lokasi": nama_lokasi(lokasi), "horizon": H, "model": nama, "split": nama_split, **metrik(asli, pred)})
    return pd.DataFrame(baris)

---
## 2. Model

Sama dengan Sesi 2, ditambah tiga pengaturan: jumlah lapisan (`layers`), ukuran hidden (`hidden`), dan `dropout`.

- **ANN**: jendela `[W, fitur]` diratakan → `layers` lapisan tersembunyi (masing-masing `hidden` neuron, ReLU) → output.
- **RNN / LSTM / GRU**: `layers` lapisan rekuren bertumpuk → *hidden state* jam terakhir → output.
- `dropout` dipasang setelah setiap lapisan tersembunyi ANN, di antara lapisan rekuren, dan sebelum lapisan output.

In [ ]:
class ANN(nn.Module):
    def __init__(self, W, n_fitur, n_out, hidden, layers, dropout):
        super().__init__()
        lapis, n_in = [nn.Flatten()], W * n_fitur
        for _ in range(layers):
            lapis += [nn.Linear(n_in, hidden), nn.ReLU(), nn.Dropout(dropout)]
            n_in = hidden
        lapis.append(nn.Linear(n_in, n_out))
        self.net = nn.Sequential(*lapis)
    def forward(self, x):
        return self.net(x)


class Rekuren(nn.Module):
    def __init__(self, jenis, n_fitur, n_out, hidden, layers, dropout):
        super().__init__()
        sel = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[jenis]
        self.rnn = sel(n_fitur, hidden, num_layers=layers, batch_first=True, dropout=dropout if layers > 1 else 0.0)
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(hidden, n_out)
    def forward(self, x):                     # x: [batch, W, fitur]
        out, _ = self.rnn(x)
        return self.head(self.drop(out[:, -1]))


def n_param(model):
    return sum(p.numel() for p in model.parameters())

---
## 3. Konfigurasi Acuan

Semua ablasi berangkat dari konfigurasi ini (= notebook 03 Sesi 2, W = 24). Satu eksperimen hanya mengubah **satu** baris.

| Hyperparameter | Kunci | Acuan | Catatan |
|---|---|---|---|
| Lokasi | `lokasi` | `"duomo"` | Bagian D: kode lokasi kelompok |
| Panjang jendela (jam) | `W` | 24 | |
| Jam yang diramal | `horizon` | 1 | |
| Ukuran hidden | `hidden` | ANN 32, RNN 68, LSTM 33, GRU 38 | acuan dipilih agar parameter rekuren ≈ 5,5 ribu |
| Jumlah lapisan | `layers` | ANN 2, rekuren 1 | |
| Kelompok fitur | `fitur` | `"semua"` | |
| *Learning rate* | `lr` | 0,001 | |
| *Batch size* | `batch` | 64 | |
| Dropout | `dropout` | 0 | |
| Optimizer | `optimizer` | `"adam"` | `"adam"`, `"adamw"` (weight decay 1e-4), `"sgd"` (momentum 0,9) |
| Fungsi loss | `loss` | `"mse"` | `"mse"`, `"mae"`, `"huber"` |
| *Gradient clipping* | `clip` | `None` | contoh: 1.0 |
| *Early stopping patience* | `patience` | 30 | |
| Epoch maksimum | `max_epochs` | 300 | |

In [ ]:
ACUAN = dict(lokasi="duomo", W=24, horizon=1, hidden=None, layers=None, fitur="semua", lr=1e-3, batch=64,
             dropout=0.0, optimizer="adam", loss="mse", clip=None, patience=30, max_epochs=300)
HIDDEN_ACUAN = {"ANN": 32, "RNN": 68, "LSTM": 33, "GRU": 38}
LAYERS_ACUAN = {"ANN": 2, "RNN": 1, "LSTM": 1, "GRU": 1}

def lengkapi(cfg, model_nama):
    """hidden/layers = None berarti nilai acuan model tersebut."""
    cfg = {**ACUAN, **cfg}
    if cfg["hidden"] is None:
        cfg["hidden"] = HIDDEN_ACUAN[model_nama]
    if cfg["layers"] is None:
        cfg["layers"] = LAYERS_ACUAN[model_nama]
    if CEPAT:
        cfg["max_epochs"] = min(cfg["max_epochs"], 20)
    return cfg

def buat_model(model_nama, cfg, n_fitur):
    if model_nama == "ANN":
        return ANN(cfg["W"], n_fitur, cfg["horizon"], cfg["hidden"], cfg["layers"], cfg["dropout"])
    return Rekuren(model_nama, n_fitur, cfg["horizon"], cfg["hidden"], cfg["layers"], cfg["dropout"])

pd.DataFrame({m: {"param": n_param(buat_model(m, lengkapi({}, m), 7)), **{k: lengkapi({}, m)[k] for k in ["hidden", "layers"]}}
              for m in MODELS})

---
## 4. Training & Pencatatan

`latih(model_nama, seed, cfg)` melatih satu model dengan *early stopping* (bobot terbaik = val loss terendah) lalu mengukur MAE/RMSE/MAPE pada skala asli di split **val, test, dan natal**.

`jalankan(bagian, faktor, nilai)` menjalankan semua kombinasi **nilai × model × seed**, mengubah hanya `faktor` dari acuan. Setiap run langsung ditambahkan ke CSV. Run yang sudah tercatat dilewati, dan konfigurasi yang sama persis dari bagian lain disalin tanpa dilatih ulang.

In [ ]:
def latih(model_nama, seed, cfg):
    torch.manual_seed(seed)
    np.random.seed(seed)
    d = siapkan(cfg["lokasi"], cfg["fitur"])
    W, H = cfg["W"], cfg["horizon"]
    Xtr, ytr, _ = buat_jendela(d, W, "train", H)
    Xva, yva, _ = buat_jendela(d, W, "val", H)
    Xtr, ytr, Xva, yva = Xtr.to(DEVICE), ytr.to(DEVICE), Xva.to(DEVICE), yva.to(DEVICE)

    model = buat_model(model_nama, cfg, Xtr.shape[-1]).to(DEVICE)
    if cfg["optimizer"] == "adam":
        opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
    elif cfg["optimizer"] == "adamw":
        opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=1e-4)
    elif cfg["optimizer"] == "sgd":
        opt = torch.optim.SGD(model.parameters(), lr=cfg["lr"], momentum=0.9)
    loss_fn = {"mse": nn.MSELoss(), "mae": nn.L1Loss(), "huber": nn.HuberLoss(delta=1.0)}[cfg["loss"]]
    val_fn = nn.MSELoss()           # early stopping selalu memakai val MSE agar adil antar fungsi loss
    g = torch.Generator().manual_seed(seed)

    terbaik, state_terbaik, sabar, epoch = float("inf"), None, 0, 0
    t0 = time.time()
    for epoch in range(1, cfg["max_epochs"] + 1):
        model.train()
        for b in torch.randperm(len(Xtr), generator=g).split(cfg["batch"]):
            opt.zero_grad()
            loss = loss_fn(model(Xtr[b]), ytr[b])
            loss.backward()
            if cfg["clip"]:
                nn.utils.clip_grad_norm_(model.parameters(), cfg["clip"])
            opt.step()
        model.eval()
        with torch.no_grad():
            va = val_fn(model(Xva), yva).item()
        if va < terbaik:
            terbaik, sabar = va, 0
            state_terbaik = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            sabar += 1
            if sabar >= cfg["patience"]:
                break
    detik = time.time() - t0
    if state_terbaik is not None:   # bila loss NaN sejak awal, pakai bobot terakhir
        model.load_state_dict(state_terbaik)

    hasil = {"param": n_param(model), "epoch": epoch, "detik": round(detik, 2)}
    model.eval()
    for nama_split in ["val", "test", "natal"]:
        X, _, awal = buat_jendela(d, W, nama_split, H)
        with torch.no_grad():
            z = model(X.to(DEVICE)).cpu().numpy()
        pred = z * d["std"] + d["mean"]
        asli = d["net"][awal[:, None] + np.arange(H)]
        for k, v in metrik(asli, pred).items():
            hasil[f"{nama_split}_{k}"] = v
    return hasil

In [ ]:
KOLOM_CFG = list(ACUAN)

def baca_hasil():
    return pd.read_csv(CSV_HASIL, dtype={"nilai": str, "kunci": str}) if CSV_HASIL.exists() else pd.DataFrame()

def _kunci(model_nama, seed, cfg):
    return json.dumps([model_nama, seed] + [cfg[k] for k in KOLOM_CFG], default=str)

def jalankan(bagian, faktor, nilai, models=MODELS, seeds=None, **tetap):
    """Contoh: jalankan("A1", "W", [6, 12, 24, 48, 168]).
    `tetap` = pengaturan lain yang ikut diubah untuk semua run, contoh: jalankan("B7", "clip", [None, 1.0], lr=1e-2).
    Konfigurasi yang sudah pernah dilatih (di bagian mana pun) tidak dilatih ulang: barisnya disalin."""
    seeds = SEEDS if seeds is None else seeds
    lama = baca_hasil()
    sudah = set(zip(lama["bagian"], lama["kunci"])) if len(lama) else set()
    pernah = lama.drop_duplicates("kunci").set_index("kunci") if len(lama) else None
    for v in nilai:
        for m in models:
            for seed in seeds:
                ubah = {**tetap} if faktor is None else {**tetap, faktor: v}
                cfg = lengkapi(ubah, m)
                kunci = _kunci(m, seed, cfg)
                if (bagian, kunci) in sudah:
                    continue
                if pernah is not None and kunci in pernah.index:
                    r, asal = pernah.loc[kunci, ["param", "epoch", "detik"] + [c for c in pernah.columns if c[:4] in ("val_", "test", "nata")]].to_dict(), "disalin"
                else:
                    r, asal = latih(m, seed, cfg), "dilatih"
                baris = {"kunci": kunci, "bagian": bagian, "faktor": faktor or "-", "nilai": str(v), "model": m, "seed": seed,
                         **{k: cfg[k] for k in KOLOM_CFG}, **r}
                pd.DataFrame([baris]).to_csv(CSV_HASIL, mode="a", header=not CSV_HASIL.exists(), index=False)
                sudah.add((bagian, kunci))
                print(f"[{bagian}] {faktor or 'konfigurasi'}={v!s:>12} {m:4s} seed={seed}  epoch={int(r['epoch']):3d}  {r['detik']:6.1f} s  "
                      f"val MAE={r['val_MAE']:7.1f}  test MAE={r['test_MAE']:7.1f}  ({asal})")
    df = baca_hasil()
    return df[df.bagian == bagian]


def jalankan_konfigurasi(bagian, cfg_per_model, seeds=None, **tetap):
    """Satu konfigurasi khusus per model, contoh: jalankan_konfigurasi("C", TERBAIK)."""
    for m, cfg in cfg_per_model.items():
        jalankan(bagian, None, ["-"], models=[m], seeds=seeds, **{**cfg, **tetap})
    df = baca_hasil()
    return df[df.bagian == bagian]


def ringkas(df, kolom=("val_MAE", "test_MAE", "natal_MAE")):
    """Rata-rata ± simpangan baku antar seed untuk setiap (nilai, model)."""
    g = df.groupby(["faktor", "nilai", "model"], sort=False)
    out = g[list(kolom)].agg(["mean", "std"])
    out.columns = [f"{a}_{b}" for a, b in out.columns]
    out.insert(0, "param", g["param"].first())
    out.insert(1, "epoch", g["epoch"].mean())
    out.insert(2, "n_seed", g["seed"].nunique())
    return out.round(1)


def plot_faktor(df, judul, log_x=False, simpan=None):
    """MAE val (kiri) dan test (kanan) terhadap nilai faktor, satu garis per model, area = ± 1 sd."""
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
    urutan = list(dict.fromkeys(df["nilai"]))
    for ax, split in zip(axes, ["val", "test"]):
        for m in MODELS:
            r = df[df.model == m].groupby("nilai", sort=False)[f"{split}_MAE"].agg(["mean", "std"]).reindex(urutan).dropna(how="all")
            if r.empty:
                continue
            x = np.arange(len(r))
            ax.plot(x, r["mean"], "o-", color=WARNA_MODEL[m], label=m)
            ax.fill_between(x, r["mean"] - r["std"].fillna(0), r["mean"] + r["std"].fillna(0), color=WARNA_MODEL[m], alpha=0.15)
            ax.set_xticks(x, r.index)
        ax.set_title(f"{split} MAE")
        ax.set_xlabel(judul)
    axes[0].set_ylabel("MAE (lebih kecil = lebih baik)")
    axes[1].legend(fontsize=9)
    plt.tight_layout()
    if simpan:
        plt.savefig(HASIL_DIR / simpan, dpi=200, bbox_inches="tight")
    plt.show()

---
## Bagian 0 — Acuan & Baseline Naif

Jalankan konfigurasi acuan dan hitung baseline naif. Angka ini menjadi titik pembanding untuk seluruh tugas. Hasil acuan seharusnya mendekati notebook 03 Sesi 2 (test MAE: ANN ≈ 310, RNN ≈ 230, LSTM ≈ 274, GRU ≈ 237).

In [ ]:
naif = pd.concat([baseline_naif("duomo", 1), baseline_naif("duomo", 24)])
naif.to_csv(HASIL_DIR / "baseline_naif_duomo.csv", index=False)
naif.pivot_table(index=["horizon", "model"], columns="split", values="MAE").round(1)

In [ ]:
acuan = jalankan("0", None, ["acuan"])
ringkas(acuan)

---
## Bagian A — Ablasi Wajib (5 faktor)

Setiap sel di bawah mengubah **satu** faktor. Nilai acuan ikut dicantumkan agar setiap grafik lengkap; run yang konfigurasinya sudah pernah dilatih (misalnya acuan dari Bagian 0) tidak dilatih ulang, cukup disalin.

Setelah setiap sel, tulis di laporan: **pola apa yang terlihat, model mana yang paling sensitif, dan mengapa.** Petunjuk pertanyaan ada di setiap subbagian.

### A1. Panjang jendela `W` ∈ {6, 12, 24, 48, 168} jam

- Apakah jendela lebih panjang selalu lebih baik? Bandingkan dengan hasil "pengaruh per jam" di Sesi 2.
- Perhatikan **jumlah parameter ANN** yang membengkak seiring W, sedangkan model rekuren tetap. Apa akibatnya?

In [ ]:
a1 = jalankan("A1", "W", [6, 12, 24, 48, 168])
plot_faktor(a1, "W (jam)", simpan="A1_W.png")
ringkas(a1)

### A2. Ukuran hidden ∈ {16, 32, 64, 128}

Nilai yang sama dipakai untuk keempat model (untuk ANN: neuron per lapisan tersembunyi). Akibatnya jumlah parameter **tidak lagi setara**: LSTM dengan hidden 64 punya ± 4× parameter RNN dengan hidden 64.

- Apakah model yang lebih besar selalu lebih baik pada data sekecil ini (672 sampel train)?
- Buat juga grafik MAE terhadap **jumlah parameter** (kolom `param`) dan bandingkan dengan grafik terhadap ukuran hidden.

In [ ]:
a2 = jalankan("A2", "hidden", [16, 32, 64, 128])
plot_faktor(a2, "ukuran hidden", simpan="A2_hidden.png")
ringkas(a2)

### A3. Jumlah lapisan ∈ {1, 2, 3}

Acuan: ANN 2 lapisan, model rekuren 1 lapisan. Lapisan tambahan memakai ukuran hidden acuan masing-masing model.

- Apakah menumpuk lapisan rekuren membantu? Lihat juga kolom `epoch`: apakah model yang lebih dalam lebih cepat *overfit*?

In [ ]:
a3 = jalankan("A3", "layers", [1, 2, 3])
plot_faktor(a3, "jumlah lapisan", simpan="A3_layers.png")
ringkas(a3)

### A4. *Learning rate* ∈ {0,0001, 0,001, 0,01}

- Learning rate kecil: apakah training berhenti karena `max_epochs` sebelum konvergen? (lihat kolom `epoch` mendekati 300)
- Learning rate besar: apakah hasilnya lebih tidak stabil antar seed (simpangan baku besar)?

In [ ]:
a4 = jalankan("A4", "lr", [1e-4, 1e-3, 1e-2])
plot_faktor(a4, "learning rate", simpan="A4_lr.png")
ringkas(a4)

### A5. Kelompok fitur ∈ {`net`, `net+call+sms`, `net+waktu`, `semua`}

- Fitur mana yang paling berguna: trafik telepon/SMS atau informasi jam & hari?
- Apakah model rekuren lebih membutuhkan fitur waktu dibanding ANN, atau sebaliknya? Kaitkan dengan cara masing-masing model membaca urutan.

In [ ]:
a5 = jalankan("A5", "fitur", ["net", "net+call+sms", "net+waktu", "semua"])
plot_faktor(a5, "kelompok fitur", simpan="A5_fitur.png")
ringkas(a5)

---
## Bagian B — Ablasi Pilihan (minimal 2 faktor)

Pilih **minimal dua** faktor berikut. Hapus tanda `#` pada sel yang dipilih. Tulis di laporan alasan memilih faktor tersebut dan hipotesis kalian **sebelum** menjalankan.

| Kode | Faktor | Nilai | Pertanyaan pemandu |
|---|---|---|---|
| B1 | `batch` | 16, 64, 256 | Batch kecil = update lebih sering tetapi lebih berisik. Mana yang lebih baik dan lebih cepat? |
| B2 | `dropout` | 0, 0,2, 0,5 | Apakah regularisasi membantu pada data sekecil ini? Coba juga dengan `layers=2`. |
| B3 | `optimizer` | adam, adamw, sgd | Apakah SGD butuh learning rate berbeda? |
| B4 | `loss` | mse, mae, huber | Loss mana yang paling baik untuk **metrik MAE**? Bagaimana dengan RMSE? |
| B5 | `patience` | 10, 30, 60 | Seberapa sensitif hasil terhadap kapan training dihentikan? |
| B6 | `horizon` | 1, 6, 24 | Pada horizon berapa model mulai kalah dari baseline naif "minggu lalu"? Bandingkan dengan tabel Bagian 0. |
| B7 | `clip` | None, 0,5, 1,0 | Apakah *gradient clipping* menstabilkan RNN, terutama dengan learning rate 0,01? |

In [ ]:
# b1 = jalankan("B1", "batch", [16, 64, 256]);              plot_faktor(b1, "batch size", simpan="B1_batch.png");     display(ringkas(b1))
# b2 = jalankan("B2", "dropout", [0.0, 0.2, 0.5]);           plot_faktor(b2, "dropout", simpan="B2_dropout.png");      display(ringkas(b2))
# b3 = jalankan("B3", "optimizer", ["adam", "adamw", "sgd"]); plot_faktor(b3, "optimizer", simpan="B3_optimizer.png"); display(ringkas(b3))
# b4 = jalankan("B4", "loss", ["mse", "mae", "huber"]);       plot_faktor(b4, "fungsi loss", simpan="B4_loss.png");    display(ringkas(b4))
# b5 = jalankan("B5", "patience", [10, 30, 60]);              plot_faktor(b5, "patience", simpan="B5_patience.png");   display(ringkas(b5))
# b6 = jalankan("B6", "horizon", [1, 6, 24]);                 plot_faktor(b6, "horizon (jam)", simpan="B6_horizon.png"); display(ringkas(b6))
# b7 = jalankan("B7", "clip", [None, 0.5, 1.0], lr=1e-2);     plot_faktor(b7, "clip (lr = 0,01)", simpan="B7_clip.png"); display(ringkas(b7))

---
## Bagian C — Konfigurasi Terbaik per Model

1. Untuk **setiap model**, gabungkan nilai terbaik dari Bagian A dan B berdasarkan **val MAE rata-rata** (bukan test!).
2. Isi `TERBAIK` di bawah. Cukup tulis kunci yang berbeda dari acuan. Biarkan `horizon` = 1 agar sebanding dengan acuan.
3. Gabungan nilai terbaik belum tentu terbaik bersama-sama (faktor bisa saling memengaruhi). Bila val MAE gabungan lebih buruk dari acuan, boleh mencoba beberapa kombinasi lagi dengan label bagian `"C-coba1"`, `"C-coba2"`, dst. Semua percobaan tetap dilaporkan.

In [ ]:
# CONTOH format — ganti dengan temuan kalian sendiri
TERBAIK = {
    "ANN":  dict(),
    "RNN":  dict(),
    "LSTM": dict(),
    "GRU":  dict(),
}
c = jalankan_konfigurasi("C", TERBAIK)
banding = pd.concat([acuan.assign(konfigurasi="acuan"), c.assign(konfigurasi="terbaik")])
tabel_c = banding.groupby(["model", "konfigurasi"])[["param", "val_MAE", "test_MAE", "test_MAPE", "natal_MAE"]].agg(["mean", "std"]).round(1)
tabel_c.to_csv(HASIL_DIR / "C_terbaik_vs_acuan.csv")
tabel_c

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
x, lebar = np.arange(len(MODELS)), 0.38
for j, (nama, df, alpha) in enumerate([("acuan", acuan, 0.5), ("terbaik", c, 1.0)]):
    r = df.groupby("model")["test_MAE"].agg(["mean", "std"]).reindex(MODELS)
    ax.bar(x + (j - 0.5) * lebar, r["mean"], lebar, yerr=r["std"], capsize=4, alpha=alpha,
           color=[WARNA_MODEL[m] for m in MODELS], label=nama)
for nama, gaya in {"Naif: nilai terakhir": "-.", "Naif: kemarin": "--", "Naif: minggu lalu": ":"}.items():
    v = naif.query("horizon == 1 and split == 'test' and model == @nama").MAE.item()
    ax.axhline(v, color=GRAY, ls=gaya, lw=1.3, label=f"{nama} ({v:.0f})")
ax.set_xticks(x, MODELS); ax.set_ylabel("test MAE"); ax.legend(fontsize=8, ncol=2)
ax.set_title("Acuan vs konfigurasi terbaik — minggu test, Duomo")
plt.tight_layout(); plt.savefig(HASIL_DIR / "C_terbaik_vs_acuan.png", dpi=200, bbox_inches="tight"); plt.show()

---
## Bagian D — Uji Generalisasi di Lokasi Kelompok

Latih ulang konfigurasi **acuan** dan **terbaik** (dari Bagian C, tanpa diubah) di lokasi kelompok kalian, lalu bandingkan dengan baseline naif lokasi tersebut.

- Apakah urutan model (siapa terbaik) sama dengan di Duomo?
- Apakah konfigurasi terbaik Duomo tetap lebih baik dari acuan di lokasi kalian? Jika tidak, apa bedanya pola trafik lokasi kalian (lihat grafik di bawah)?
- Bagaimana dengan minggu Natal di lokasi kalian?

In [ ]:
dfk, dfd = muat_data(LOKASI_KELOMPOK), muat_data("duomo")
fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
for ax, df, nama, warna in [(axes[0], dfd, "Duomo", NAVY), (axes[1], dfk, nama_lokasi(LOKASI_KELOMPOK), TEAL)]:
    ax.plot(df.index, df.net, color=warna, lw=0.8)
    ax.set_title(f"{nama}: aktivitas internet per jam", fontsize=10)
plt.tight_layout(); plt.savefig(HASIL_DIR / "D_pola_lokasi.png", dpi=200, bbox_inches="tight"); plt.show()

naif_k = baseline_naif(LOKASI_KELOMPOK, 1)
naif_k.to_csv(HASIL_DIR / "baseline_naif_lokasi.csv", index=False)
naif_k.pivot_table(index="model", columns="split", values="MAE").round(1)

In [ ]:
d_acuan = jalankan("D-acuan", "lokasi", [LOKASI_KELOMPOK])
d_terbaik = jalankan_konfigurasi("D-terbaik", TERBAIK, lokasi=LOKASI_KELOMPOK)
banding_d = pd.concat([d_acuan.assign(konfigurasi="acuan"), d_terbaik.assign(konfigurasi="terbaik")])
tabel_d = banding_d.groupby(["model", "konfigurasi"])[["val_MAE", "test_MAE", "test_MAPE", "natal_MAE"]].agg(["mean", "std"]).round(1)
tabel_d.to_csv(HASIL_DIR / "D_generalisasi.csv")
print("Lokasi:", nama_lokasi(LOKASI_KELOMPOK))
tabel_d

---
## Penutup — Cek Sebelum Mengumpulkan

- [ ] `CEPAT = False` dan semua sel sudah dijalankan berurutan tanpa error.
- [ ] `hasil/ablasi.csv` berisi seluruh run (3 seed per konfigurasi).
- [ ] Setiap angka dan grafik di laporan bisa dilacak ke `hasil/` di repo.
- [ ] Notebook di-commit **beserta output-nya**.
- [ ] README repo memuat nama & NIM seluruh anggota, kelas, dan kode lokasi.

In [ ]:
semua = baca_hasil()
print(len(semua), "run tercatat |", semua.groupby("bagian").size().to_dict())
print("Total waktu training: {:.1f} menit".format(semua.drop_duplicates("kunci").detik.sum() / 60))